In [ ]:
git!pip install -q langchain langchain-groq langchain-huggingface langchain-community faiss-cpu pymupdf tabula-py
!apt-get install -y -q default-jre
print("All packages installed")

In [ ]:
import os
import json
import fitz
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
os.environ["GROQ_API_KEY"] = "your_actual_key_here"
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
print("Imports done — using llama-3.3-70b-versatile")

In [ ]:
CUSTOMS_RULES = """
HS CODE RULES:
- HS codes must be 6-10 digits
- Chapter 84-85: Machinery and electrical equipment (laptops, phones)
- Chapter 61-62: Clothing and apparel
- Chapter 01-05: Live animals and animal products
- HS code must match the goods description

VALUE RULES:
- Declared value must match the commercial invoice value
- Currency must be clearly stated
- Shipments over USD 2500 require formal customs entry in USA
- Undervaluation is a serious compliance violation

REQUIRED FIELDS:
- Shipper name and country are mandatory
- Consignee name and country are mandatory
- Country of origin is mandatory for all goods
- Bill of lading number must be present
- Incoterms must be specified

COMMON ERRORS:
- Missing consignee contact information
- HS code mismatch with goods description
- Declared value in wrong or missing currency
- Missing country of origin especially for textiles
- No bill of lading reference number
"""

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.create_documents([CUSTOMS_RULES])

# Build vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
kb = FAISS.from_documents(chunks, embeddings)

print(f"Knowledge base ready — {len(chunks)} chunks indexed")

In [ ]:
def parse_document(file_path: str) -> str:
    """Extract all text from a PDF or text file."""

    if file_path.endswith(".txt"):
        with open(file_path, "r") as f:
            return f.read()

    # PDF extraction with PyMuPDF
    text = ""
    doc = fitz.open(file_path)
    for page in doc:
        text += page.get_text()
    doc.close()

    return text.strip()

print("Parser ready")

In [ ]:
EXTRACT_PROMPT = ChatPromptTemplate.from_template("""
You are a customs document expert. Extract fields from this shipping document.

DOCUMENT:
{text}

Return ONLY a JSON object with these exact keys (use null if not found):
{{
  "shipper_name": null,
  "shipper_country": null,
  "consignee_name": null,
  "consignee_country": null,
  "hs_tariff_code": null,
  "goods_description": null,
  "declared_value": null,
  "currency": null,
  "weight_kg": null,
  "country_of_origin": null,
  "incoterms": null,
  "bill_of_lading_number": null
}}

Return ONLY the JSON. No explanation.
""")

def extract_fields(text: str) -> dict:
    chain = EXTRACT_PROMPT | llm
    response = chain.invoke({"text": text})
    raw = response.content.strip()

    # Strip markdown fences if present
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    try:
        return json.loads(raw.strip())
    except:
        return {"parse_error": True, "raw": response.content}

print("Field extractor ready")

In [ ]:
COMPLIANCE_PROMPT = ChatPromptTemplate.from_template("""
You are a customs compliance officer. Use the rules below to review this shipment.

CUSTOMS RULES:
{rules}

SHIPMENT DATA:
{fields}

Return ONLY a JSON object:
{{
  "issues": ["serious problems that must be fixed"],
  "warnings": ["minor concerns to review"],
  "passed_checks": ["fields and rules that are correct"],
  "risk_level": "LOW or MEDIUM or HIGH"
}}

Return ONLY the JSON. No explanation.
""")

def check_compliance(fields: dict) -> dict:
    # Retrieve relevant rules from knowledge base
    docs = kb.similarity_search(json.dumps(fields), k=4)
    rules = "\n\n".join([d.page_content for d in docs])

    chain = COMPLIANCE_PROMPT | llm
    response = chain.invoke({
        "rules": rules,
        "fields": json.dumps(fields, indent=2)
    })

    raw = response.content.strip()
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    try:
        return json.loads(raw.strip())
    except:
        return {"parse_error": True, "raw": response.content}

print("Compliance checker ready")

In [ ]:
def run_agent(file_path: str):
    print(f"📄 Processing: {file_path}\n")

    print("Step 1/3 — Parsing document...")
    text = parse_document(file_path)

    print("Step 2/3 — Extracting fields with Groq...")
    fields = extract_fields(text)

    print("Step 3/3 — Checking compliance with RAG...\n")
    compliance = check_compliance(fields)

    print("=" * 50)
    print("📦 EXTRACTED FIELDS:")
    print(json.dumps(fields, indent=2))
    print("\n📋 COMPLIANCE REPORT:")
    print(json.dumps(compliance, indent=2))
    print("=" * 50)

    return {"fields": fields, "compliance": compliance}

print("Pipeline ready")

In [ ]:
# Create a sample shipping document to test with
sample_doc = """
COMMERCIAL INVOICE

Shipper: TechCorp Shanghai Ltd, Shanghai, China
Consignee: GlobalImports Inc, New York, USA

Bill of Lading: BL-2024-00123
Date: 2024-01-15

Description: Laptop computers x 50 units
HS Code: 8471.30
Unit Price: USD 800
Total Declared Value: USD 40,000
Weight: 125 kg
Country of Origin: China
Incoterms: FOB Shanghai
"""

with open("test_invoice.txt", "w") as f:
    f.write(sample_doc)

# Run the full agent
result = run_agent("test_invoice.txt")

In [ ]:
CUSTOMS_RULES_EXPANDED = """
HS CODE RULES:
- HS codes must be 6-10 digits
- Chapter 84-85: Machinery and electrical equipment (laptops, phones, computers)
- Chapter 61-62: Clothing and apparel
- Chapter 01-05: Live animals and animal products
- Chapter 27: Mineral fuels and oils
- Chapter 30: Pharmaceutical products
- Chapter 72-73: Iron and steel products
- HS code must match the goods description exactly
- Mismatched HS codes can result in shipment delays and penalties

VALUE RULES:
- Declared value must match the commercial invoice value exactly
- Currency must be clearly stated on all documents
- Shipments over USD 2500 require formal customs entry in USA
- Shipments over EUR 1000 require customs declaration in EU
- Undervaluation is a serious compliance violation and can result in seizure
- Insurance and freight costs must be included in CIF valuations

REQUIRED FIELDS:
- Shipper full name and country are mandatory
- Consignee full name and country are mandatory
- Country of origin is mandatory for all goods
- Bill of lading or airway bill number must be present
- Incoterms must be specified (FOB, CIF, EXW, DDP etc)
- Net and gross weight must be declared
- Number of packages must be stated

PROHIBITED AND RESTRICTED GOODS:
- Dangerous goods require IATA or IMDG special documentation
- Lithium batteries require UN3480 or UN3481 declaration
- Food products require health certificates
- Plants and wood products require phytosanitary certificates
- Weapons and dual-use goods require export licenses

COMMON ERRORS:
- Missing consignee contact information
- HS code mismatch with goods description
- Declared value in wrong or missing currency
- Missing country of origin especially for textiles
- No bill of lading or airway bill reference number
- Incoterms not specified or incorrectly stated
- Weight missing or inconsistent across documents
- No package count declared
- Lithium battery goods missing UN declaration
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.create_documents([CUSTOMS_RULES_EXPANDED])
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
kb = FAISS.from_documents(chunks, embeddings)

print(f"Expanded knowledge base ready — {len(chunks)} chunks indexed")

In [ ]:
EXTRACT_PROMPT = ChatPromptTemplate.from_template("""
You are a senior customs document expert with 20 years experience.
Extract every available field from this shipping document carefully.

DOCUMENT:
{text}

Return ONLY a JSON object with these exact keys (use null if not found):
{{
  "shipper_name": null,
  "shipper_country": null,
  "shipper_address": null,
  "consignee_name": null,
  "consignee_country": null,
  "consignee_address": null,
  "hs_tariff_code": null,
  "goods_description": null,
  "quantity": null,
  "declared_value": null,
  "currency": null,
  "weight_kg": null,
  "package_count": null,
  "country_of_origin": null,
  "incoterms": null,
  "bill_of_lading_number": null,
  "vessel_or_flight": null,
  "port_of_loading": null,
  "port_of_discharge": null,
  "contains_lithium_battery": null,
  "dangerous_goods": null
}}

Return ONLY the JSON. No explanation. No markdown.
""")

def extract_fields(text: str) -> dict:
    chain = EXTRACT_PROMPT | llm
    response = chain.invoke({"text": text})
    raw = response.content.strip()
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except:
        return {"parse_error": True, "raw": response.content}

print(" Upgraded field extractor ready")

In [ ]:
COMPLIANCE_PROMPT = ChatPromptTemplate.from_template("""
You are a senior customs compliance officer. Use the rules below to review this shipment carefully.

CUSTOMS RULES:
{rules}

SHIPMENT DATA:
{fields}

Check every field against the rules. Be thorough.

Return ONLY a JSON object with exactly this structure:
{{
  "issues": ["list of serious problems that will cause delays or penalties"],
  "warnings": ["list of minor concerns to review"],
  "passed_checks": ["list of fields and rules that are correct"],
  "missing_fields": ["list of fields that are null but required"],
  "risk_level": "LOW or MEDIUM or HIGH",
  "risk_score": 0,
  "recommendation": "one sentence summary of what the shipper should do"
}}

risk_score must be a number from 0 to 100.
Return ONLY the JSON. No explanation. No markdown.
""")

def check_compliance(fields: dict) -> dict:
    docs = kb.similarity_search(json.dumps(fields), k=5)
    rules = "\n\n".join([d.page_content for d in docs])
    chain = COMPLIANCE_PROMPT | llm
    response = chain.invoke({
        "rules": rules,
        "fields": json.dumps(fields, indent=2)
    })
    raw = response.content.strip()
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except:
        return {"parse_error": True, "raw": response.content}

print("Upgraded compliance checker ready")

In [ ]:
def generate_report(file_path: str, fields: dict, compliance: dict) -> str:

    risk = compliance.get("risk_level", "UNKNOWN")
    score = compliance.get("risk_score", "N/A")

    risk_symbol = {"LOW": "🟢", "MEDIUM": "🟡", "HIGH": "🔴"}.get(risk, "⚪")

    lines = []
    lines.append("=" * 60)
    lines.append("   CARGOWISE DOCUMENT INTELLIGENCE AGENT")
    lines.append("   Customs Compliance Report")
    lines.append("=" * 60)
    lines.append(f"   Document  : {file_path}")
    lines.append(f"   Risk Level: {risk_symbol} {risk}  (Score: {score}/100)")
    lines.append("=" * 60)

    lines.append("\n📦 SHIPMENT DETAILS")
    lines.append("-" * 40)
    field_labels = {
        "shipper_name":          "Shipper",
        "shipper_country":       "Shipper Country",
        "shipper_address":       "Shipper Address",
        "consignee_name":        "Consignee",
        "consignee_country":     "Consignee Country",
        "consignee_address":     "Consignee Address",
        "goods_description":     "Goods",
        "hs_tariff_code":        "HS Code",
        "quantity":              "Quantity",
        "declared_value":        "Declared Value",
        "currency":              "Currency",
        "weight_kg":             "Weight (kg)",
        "package_count":         "Packages",
        "country_of_origin":     "Origin",
        "incoterms":             "Incoterms",
        "bill_of_lading_number": "B/L Number",
        "port_of_loading":       "Port of Loading",
        "port_of_discharge":     "Port of Discharge",
        "vessel_or_flight":      "Vessel / Flight",
        "contains_lithium_battery": "Lithium Battery",
        "dangerous_goods":       "Dangerous Goods",
    }
    for key, label in field_labels.items():
        value = fields.get(key)
        display = str(value) if value is not None else "⚠️  NOT FOUND"
        lines.append(f"  {label:<25} {display}")

    if compliance.get("issues"):
        lines.append("\n🔴 ISSUES — Must fix before shipping")
        lines.append("-" * 40)
        for issue in compliance["issues"]:
            lines.append(f"  ✗ {issue}")

    if compliance.get("warnings"):
        lines.append("\n🟡 WARNINGS — Review before shipping")
        lines.append("-" * 40)
        for warning in compliance["warnings"]:
            lines.append(f"  ⚠ {warning}")

    if compliance.get("missing_fields"):
        lines.append("\n⚪ MISSING FIELDS")
        lines.append("-" * 40)
        for field in compliance["missing_fields"]:
            lines.append(f"  - {field}")

    if compliance.get("passed_checks"):
        lines.append("\n🟢 PASSED CHECKS")
        lines.append("-" * 40)
        for check in compliance["passed_checks"]:
            lines.append(f"  ✓ {check}")

    lines.append("\n💡 RECOMMENDATION")
    lines.append("-" * 40)
    lines.append(f"  {compliance.get('recommendation', 'No recommendation generated.')}")

    lines.append("\n" + "=" * 60)
    lines.append("   Generated by CargoWise Document Intelligence Agent")
    lines.append("=" * 60)

    return "\n".join(lines)

print("Report generator ready")

In [ ]:
def run_agent(file_path: str):
    print(f"\n📄 Processing: {file_path}\n")

    print("Step 1/3 — Parsing document...")
    text = parse_document(file_path)

    print("Step 2/3 — Extracting fields with Groq LLM...")
    fields = extract_fields(text)

    print("Step 3/3 — Checking compliance with RAG...\n")
    compliance = check_compliance(fields)

    report = generate_report(file_path, fields, compliance)
    print(report)

    return {"fields": fields, "compliance": compliance, "report": report}

print(" Full pipeline ready")

In [ ]:
# This document is intentionally incomplete to show the agent catching errors
tricky_doc = """
COMMERCIAL INVOICE & PACKING LIST

Shipper: ShenZhen Electronics Co., Shenzhen, China
Consignee: FastTech Imports LLC, Miami, Florida

Date: 2024-03-10
Bill of Lading: BL-2024-99871

Items:
- Wireless Earbuds with built-in lithium battery x 200 units
- Unit Price: USD 45
- Total Value: USD 9,000

Weight: 85 kg
Incoterms: EXW Shenzhen
"""

with open("tricky_invoice.txt", "w") as f:
    f.write(tricky_doc)

result = run_agent("tricky_invoice.txt")

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

def create_sample_pdf(filename: str):
    c = canvas.Canvas(filename, pagesize=A4)
    width, height = A4

    # Header
    c.setFont("Helvetica-Bold", 16)
    c.drawString(50, height - 50, "COMMERCIAL INVOICE & BILL OF LADING")

    c.setFont("Helvetica", 10)
    c.drawString(50, height - 75, "Document No: BL-2024-55678")
    c.drawString(50, height - 90, "Date: 15 March 2024")

    # Shipper
    c.setFont("Helvetica-Bold", 11)
    c.drawString(50, height - 130, "SHIPPER:")
    c.setFont("Helvetica", 10)
    c.drawString(50, height - 145, "Guangzhou Manufacturing Co. Ltd")
    c.drawString(50, height - 160, "88 Industrial Avenue, Guangzhou, China")
    c.drawString(50, height - 175, "Tel: +86 20 1234 5678")

    # Consignee
    c.setFont("Helvetica-Bold", 11)
    c.drawString(300, height - 130, "CONSIGNEE:")
    c.setFont("Helvetica", 10)
    c.drawString(300, height - 145, "Pacific Traders Inc.")
    c.drawString(300, height - 160, "500 Harbor Blvd, Los Angeles, USA")
    c.drawString(300, height - 175, "Tel: +1 310 987 6543")

    # Shipment details
    c.setFont("Helvetica-Bold", 11)
    c.drawString(50, height - 220, "SHIPMENT DETAILS:")
    c.setFont("Helvetica", 10)
    c.drawString(50, height - 240, "Port of Loading      : Guangzhou, China")
    c.drawString(50, height - 255, "Port of Discharge    : Los Angeles, USA")
    c.drawString(50, height - 270, "Vessel               : MSC ANNA - Voyage 124W")
    c.drawString(50, height - 285, "Incoterms            : FOB Guangzhou")
    c.drawString(50, height - 300, "Country of Origin    : China")

    # Goods table header
    c.setFont("Helvetica-Bold", 11)
    c.drawString(50, height - 340, "GOODS DESCRIPTION:")
    c.setFont("Helvetica-Bold", 10)
    c.drawString(50,  height - 360, "Description")
    c.drawString(250, height - 360, "HS Code")
    c.drawString(350, height - 360, "Qty")
    c.drawString(420, height - 360, "Unit Price")
    c.drawString(510, height - 360, "Total")

    # Goods rows
    c.setFont("Helvetica", 10)
    c.drawString(50,  height - 378, "Smartphone (with lithium battery)")
    c.drawString(250, height - 378, "8517.12.00")
    c.drawString(350, height - 378, "500 pcs")
    c.drawString(420, height - 378, "USD 120")
    c.drawString(510, height - 378, "USD 60,000")

    c.drawString(50,  height - 395, "USB Charging Cables")
    c.drawString(250, height - 395, "8544.42.90")
    c.drawString(350, height - 395, "1000 pcs")
    c.drawString(420, height - 395, "USD 3")
    c.drawString(510, height - 395, "USD 3,000")

    # Totals
    c.setFont("Helvetica-Bold", 10)
    c.drawString(50,  height - 430, "Total Declared Value : USD 63,000")
    c.drawString(50,  height - 445, "Total Weight         : 320 kg")
    c.drawString(50,  height - 460, "Total Packages       : 25 cartons")

    # Note — intentionally missing UN lithium battery declaration
    c.setFont("Helvetica-Oblique", 9)
    c.drawString(50, height - 500, "Note: Standard commercial shipment.")

    c.save()
    print(f"✅ Sample PDF created: {filename}")

create_sample_pdf("sample_shipment.pdf")
result = run_agent("sample_shipment.pdf")

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick your PDFs from desktop

# Then run like this
result1 = run_agent("Bill_of_Lading_Sample.pdf")


In [ ]:
from google.colab import files
uploaded = files.upload()  # upload invoice.pdf this time

result2 = run_agent("invoice.pdf")